# Framework Metadata — Pipeline Config & Run Log

This notebook provisions the **metadata-driven framework** tables that control the entire pipeline dynamically.

## Tables Created
| Table | Purpose |
|---|---|
| `framework.pipeline_config` | One row per source → target table mapping. Drives Bronze, Silver, and Gold automatically. |
| `framework.pipeline_run_log` | Append-only audit log. Every notebook writes a SUCCESS or FAILURE record here. |

## `pipeline_config` Column Reference
| Column | Description |
|---|---|
| `config_id` | Surrogate key |
| `source_name` | Logical name of the source (e.g. `drivers`) |
| `source_path` | ADLS subdirectory relative to Bronze root (e.g. `drivers/`) |
| `file_format` | `json`, `csv`, or `parquet` |
| `csv_header` | `true`/`false` — only used when `file_format = csv` |
| `csv_delimiter` | Column delimiter — only used when `file_format = csv` |
| `bronze_table` | Target Bronze Delta table name |
| `silver_table` | Target Silver Delta table name |
| `gold_table` | Target Gold Delta table name (optional) |
| `primary_key` | Comma-separated PK columns used for MERGE deduplication |
| `silver_load_type` | `merge` or `append` |
| `gold_load_type` | `full` (CREATE OR REPLACE) or `incremental` (MERGE via CDF) |
| `gold_agg_sql` | Aggregation SQL template executed at Gold layer (optional) |
| `partition_col` | Column to partition Bronze/Silver tables by |
| `zorder_cols` | Comma-separated columns for OPTIMIZE ZORDER |
| `is_active` | `true`/`false` — set to `false` to skip without deleting the config |
| `created_at` | Row creation timestamp |
| `updated_at` | Last modification timestamp |

In [ ]:
%run ./fw_0.config

In [ ]:
# ── Step 1: Provision Framework Schema ───────────────────────────────────────

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {FRAMEWORK_SCHEMA}
    MANAGED LOCATION '{CATALOG_ROOT}'
    COMMENT 'Metadata framework tables for the dynamic medallion pipeline'
""")
print(f"Schema ready: {CATALOG_NAME}.{FRAMEWORK_SCHEMA}")

In [ ]:
# ── Step 2: Create pipeline_config (metadata-driven control table) ────────────

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FRAMEWORK_CONFIG_TABLE} (
    config_id      INT          NOT NULL,
    source_name    STRING       NOT NULL COMMENT 'Logical name of the source dataset',
    source_path    STRING       NOT NULL COMMENT 'Subdirectory under Bronze ADLS root e.g. drivers/',
    file_format    STRING       NOT NULL COMMENT 'json | csv | parquet',
    csv_header     BOOLEAN                COMMENT 'CSV only: whether file has a header row',
    csv_delimiter  STRING                 COMMENT 'CSV only: column delimiter character',
    bronze_table   STRING       NOT NULL COMMENT 'Bronze Delta table name',
    silver_table   STRING       NOT NULL COMMENT 'Silver Delta table name',
    gold_table     STRING                COMMENT 'Gold Delta table name (NULL = no Gold target)',
    primary_key    STRING       NOT NULL COMMENT 'Comma-separated PK columns for MERGE',
    silver_load_type STRING     NOT NULL COMMENT 'merge | append',
    gold_load_type STRING                COMMENT 'full | incremental',
    gold_agg_sql   STRING                COMMENT 'Aggregation SQL template for Gold (optional)',
    partition_col  STRING                COMMENT 'Partition column for Bronze/Silver tables',
    zorder_cols    STRING                COMMENT 'Comma-separated ZORDER columns',
    is_active      BOOLEAN      NOT NULL COMMENT 'false = skip this config without deleting it',
    created_at     TIMESTAMP    NOT NULL COMMENT 'Row creation timestamp',
    updated_at     TIMESTAMP    NOT NULL COMMENT 'Last modification timestamp'
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'framework'
)
COMMENT 'Master config table — one row per source-to-target table mapping'
""")
print(f"Created: {FRAMEWORK_CONFIG_TABLE}")

In [ ]:
# ── Step 3: Create pipeline_run_log (observability audit table) ───────────────

RUN_LOG_TABLE = fq(FRAMEWORK_SCHEMA, "pipeline_run_log")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_LOG_TABLE} (
    log_id        BIGINT GENERATED ALWAYS AS IDENTITY,
    config_id     INT       NOT NULL COMMENT 'FK to pipeline_config.config_id',
    layer         STRING    NOT NULL COMMENT 'bronze | silver | gold',
    status        STRING    NOT NULL COMMENT 'SUCCESS | FAILURE | SKIPPED',
    rows_affected INT                COMMENT 'Rows inserted/updated/merged',
    error_message STRING             COMMENT 'Exception message on FAILURE',
    run_timestamp TIMESTAMP NOT NULL COMMENT 'UTC timestamp of this run'
)
USING DELTA
PARTITIONED BY (layer)
TBLPROPERTIES ('quality' = 'framework')
COMMENT 'Append-only audit log — every pipeline run writes a record here'
""")
print(f"Created: {RUN_LOG_TABLE}")

In [ ]:
# ── Step 4: Seed pipeline_config with F1 source tables ───────────────────────
# Edit this seed data to add new source tables without changing any pipeline code.

from pyspark.sql import Row
from datetime import datetime

now = datetime.utcnow()

configs = [
    Row(
        config_id      = 1,
        source_name    = "drivers",
        source_path    = "drivers/",
        file_format    = "json",
        csv_header     = None,
        csv_delimiter  = None,
        bronze_table   = "drivers",
        silver_table   = "drivers",
        gold_table     = "driver_wins",
        primary_key    = "driverId",
        silver_load_type = "merge",
        gold_load_type   = "full",
        gold_agg_sql   = (
            "SELECT d.name, d.nationality, COUNT(*) AS number_of_wins "
            "FROM {silver_drivers} d JOIN {silver_results} r ON d.driver_id = r.driver_id "
            "WHERE r.position = 1 GROUP BY d.name, d.nationality"
        ),
        partition_col  = "ingestion_date",
        zorder_cols    = "driverId",
        is_active      = True,
        created_at     = now,
        updated_at     = now
    ),
    Row(
        config_id      = 2,
        source_name    = "results",
        source_path    = "results/",
        file_format    = "json",
        csv_header     = None,
        csv_delimiter  = None,
        bronze_table   = "results",
        silver_table   = "results",
        gold_table     = None,
        primary_key    = "resultId",
        silver_load_type = "merge",
        gold_load_type   = None,
        gold_agg_sql   = None,
        partition_col  = "ingestion_date",
        zorder_cols    = "raceId,driverId",
        is_active      = True,
        created_at     = now,
        updated_at     = now
    ),
]

seed_df = spark.createDataFrame(configs)

# MERGE to make seeding idempotent — re-running won't duplicate rows
seed_df.createOrReplaceTempView("config_seed")
spark.sql(f"""
MERGE INTO {FRAMEWORK_CONFIG_TABLE} AS tgt
USING config_seed AS src
ON tgt.config_id = src.config_id
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Seeded {len(configs)} config rows into {FRAMEWORK_CONFIG_TABLE}")
spark.table(FRAMEWORK_CONFIG_TABLE).display()

In [ ]:
# ── Step 5: Grant access to framework schema ──────────────────────────────────

spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG_NAME}.{FRAMEWORK_SCHEMA} TO `data-engineers`")
spark.sql(f"GRANT SELECT ON TABLE {FRAMEWORK_CONFIG_TABLE} TO `data-engineers`")
spark.sql(f"GRANT SELECT ON TABLE {fq(FRAMEWORK_SCHEMA, 'pipeline_run_log')} TO `data-engineers`")
print("Framework GRANTs applied.")